# 05 - EDA Before Feature Engineering

Purpose:
- Inspect merged canonical laptop dataset before feature engineering.
- Focus on target distribution, source bias, missingness, category coverage, price signal, duplicate/spec overlap, and leakage/audit column review.
- Produce a compact set of recommendations for the next notebook: `06_feature_engineering_preprocessing.ipynb`.

Explicit scope:
- This notebook is not a modeling notebook.
- This notebook does not train models.
- This notebook does not fit encoders/scalers.
- This notebook only informs FE/preprocessing decisions.

## 1. Setup

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None
    plt.style.use("seaborn-v0_8-whitegrid")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "data" and PROJECT_ROOT.parent.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MERGED_PATH = PROJECT_ROOT / "data" / "intern" / "laptop_merged_cleaned.csv"
MERGE_CONFIG_PATH = PROJECT_ROOT / "docs" / "laptop_merge_config.json"
MERGE_REPORT_PATH = PROJECT_ROOT / "docs" / "laptop_merged_report.csv"
EDA_OUTPUT_DIR = PROJECT_ROOT / "data" / "eda"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures" / "eda"
EDA_SUMMARY_PATH = EDA_OUTPUT_DIR / "laptop_eda_summary.csv"
FE_RECOMMENDATION_PATH = EDA_OUTPUT_DIR / "laptop_eda_feature_recommendations.json"
EDA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not MERGED_PATH.exists():
    raise FileNotFoundError(f"Missing merged dataset: {MERGED_PATH}")
merge_config = {}
if MERGE_CONFIG_PATH.exists():
    with open(MERGE_CONFIG_PATH, "r", encoding="utf-8") as f:
        merge_config = json.load(f)
else:
    warnings.warn(f"Merge config not found: {MERGE_CONFIG_PATH}. Fallback column lists will be used.")

df = pd.read_csv(MERGED_PATH)
merge_report = pd.read_csv(MERGE_REPORT_PATH) if MERGE_REPORT_PATH.exists() else None
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Merged shape: {df.shape}")
print("Columns:")
print(list(df.columns))
print("\nDtypes:")
display(df.dtypes.to_frame("dtype"))
print("\nFirst 3 rows:")
display(df.head(3))

## 2. Helper functions

In [ ]:
SUMMARY_ROWS = []
MISSING_LIKE_TOKENS = {"unknown", "missing", "nan", "<na>", "none", ""}

def display_section(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

def safe_percent(s):
    return s.value_counts(dropna=False, normalize=True).mul(100).round(2)

def missing_like_rate(s):
    base = s.isna()
    if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype):
        normalized = s.astype("string").str.strip().str.lower()
        base = base | normalized.isin(MISSING_LIKE_TOKENS)
    return float(base.mean())

def numeric_summary_by_group(data, value_col, group_col):
    if value_col not in data or group_col not in data:
        return pd.DataFrame()
    tmp = data[[group_col, value_col]].copy()
    tmp[value_col] = pd.to_numeric(tmp[value_col], errors="coerce")
    return (tmp.groupby(group_col, dropna=False)[value_col]
        .agg(count="count", mean="mean", median="median", std="std", min="min",
             p05=lambda x: x.quantile(0.05), p25=lambda x: x.quantile(0.25),
             p75=lambda x: x.quantile(0.75), p95=lambda x: x.quantile(0.95), max="max")
        .reset_index())

def categorical_cardinality_by_source(data, cols):
    rows = []
    for col in cols:
        if col not in data:
            continue
        row = {"column": col, "nunique_overall": data[col].nunique(dropna=True)}
        if "source" in data:
            for src, part in data.groupby("source", dropna=False):
                row[f"nunique_{src}"] = part[col].nunique(dropna=True)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("nunique_overall", ascending=False)

def top_values_table(data, col, source_col="source", top_n=15):
    if col not in data:
        return pd.DataFrame()
    frames = []
    overall = data[col].fillna("<NA>").astype(str).value_counts(dropna=False).head(top_n)
    frames.append(pd.DataFrame({"scope": "overall", "value": overall.index, "n": overall.values, "pct": overall.values / len(data)}))
    if source_col in data:
        for src, part in data.groupby(source_col, dropna=False):
            vc = part[col].fillna("<NA>").astype(str).value_counts(dropna=False).head(top_n)
            frames.append(pd.DataFrame({"scope": f"source={src}", "value": vc.index, "n": vc.values, "pct": vc.values / len(part)}))
    return pd.concat(frames, ignore_index=True)

def median_price_table(data, group_cols, min_n=20):
    cols = [c for c in group_cols if c in data.columns]
    if not cols or "target_price" not in data:
        return pd.DataFrame()
    agg_spec = {
        "n": ("target_price", "size"),
        "median_target_price": ("target_price", "median"),
        "mean_target_price": ("target_price", "mean"),
        "p25": ("target_price", lambda x: x.quantile(0.25)),
        "p75": ("target_price", lambda x: x.quantile(0.75)),
    }
    if "log_target_price" in data:
        agg_spec["median_log_target_price"] = ("log_target_price", "median")
    out = data.groupby(cols, dropna=False).agg(**agg_spec).reset_index()
    out = out[out["n"] >= min_n]
    sort_cols = [c for c in ["source", "median_target_price"] if c in out]
    return out.sort_values(sort_cols, ascending=[True, False][:len(sort_cols)]) if sort_cols else out.sort_values("median_target_price", ascending=False)

def save_current_fig(name):
    path = FIGURE_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path

def append_summary(section, metric, value, source=None, notes=None):
    SUMMARY_ROWS.append({"section": section, "metric": metric, "value": value, "source": source, "notes": notes})

def plot_bar(series, title, ylabel="", filename=None):
    ax = series.plot(kind="bar")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=30)
    if filename:
        save_current_fig(filename)
    plt.show()

def numpy_safe(obj):
    if isinstance(obj, dict):
        return {str(k): numpy_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [numpy_safe(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) or np.isinf(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

## 3. Column role definition

`source` is used for EDA and evaluation, not a default baseline feature.  
`target_price` and `log_target_price` are labels, never features.  
`target_price_source` is audit only.

In [ ]:
fallback_label_columns = ["target_price", "log_target_price"]
fallback_audit_columns = ["source", "source_row_id", "merged_row_id", "url", "title", "target_price_source", "target_price_is_outlier", "target_price_missing", "merged_spec_key", "is_cross_source_soft_duplicate"]
label_columns = merge_config.get("label_columns", fallback_label_columns)
audit_columns = merge_config.get("audit_columns", fallback_audit_columns)
canonical_columns = merge_config.get("canonical_columns", list(df.columns))
leakage_columns = merge_config.get("leakage_columns", {})

numeric_candidates = ["ram_gb", "storage_gb", "screen_size_inch", "warranty_months"]
categorical_candidates = ["brand_clean", "brand_grouped", "model_clean", "model_grouped", "cpu_brand", "cpu_tier", "gpu_tier", "storage_type_clean", "condition_clean", "warranty_status", "origin_clean"]
boolean_candidates = ["brand_is_rare", "model_is_rare", "ram_missing", "storage_missing", "screen_missing", "cpu_missing", "gpu_missing", "ram_suspicious", "storage_suspicious", "screen_suspicious", "potential_dedicated_gpu", "repair_mismatch", "flag_price_spread_warn", "flag_price_spread_critical", "is_soft_duplicate_spec"]
existing_numeric_candidates = [c for c in numeric_candidates if c in df.columns]
existing_categorical_candidates = [c for c in categorical_candidates if c in df.columns]
existing_boolean_candidates = [c for c in boolean_candidates if c in df.columns]
existing_label_cols = [c for c in label_columns if c in df.columns]
existing_audit_cols = [c for c in audit_columns if c in df.columns]
print("Label columns:", existing_label_cols)
print("Audit columns:", existing_audit_cols)
print("Numeric candidates:", existing_numeric_candidates)
print("Categorical candidates:", existing_categorical_candidates)
print("Boolean candidates:", existing_boolean_candidates)

## 4. Basic data health

In [ ]:
display_section("Basic data health")
required = ["target_price", "log_target_price", "source"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns are missing: {missing_required}")
health = {
    "rows": len(df),
    "columns": df.shape[1],
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicated_merged_row_id": int(df["merged_row_id"].duplicated().sum()) if "merged_row_id" in df else None,
    "unique_sources": int(df["source"].nunique(dropna=True)),
    "source_values": sorted(df["source"].dropna().unique().tolist()),
    "target_missing_count": int(df["target_price"].isna().sum()),
    "target_non_positive_count": int((pd.to_numeric(df["target_price"], errors="coerce") <= 0).sum()),
}
numeric_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
health["numeric_infinite_values"] = int(np.isinf(df[numeric_cols_all].to_numpy()).sum()) if numeric_cols_all else 0
canonical_completeness = pd.Series({c: 1 - missing_like_rate(df[c]) for c in canonical_columns if c in df}).sort_values()
display(pd.DataFrame([health]).T.rename(columns={0: "value"}))
display(canonical_completeness.to_frame("completeness").head(20))
for metric, value in health.items():
    append_summary("basic_health", metric, value)
append_summary("basic_health", "lowest_canonical_completeness", canonical_completeness.head(10).to_dict())

## 5. Source composition

In [ ]:
display_section("Source composition")
source_counts = df["source"].value_counts(dropna=False).rename("n").to_frame()
source_counts["pct"] = source_counts["n"] / len(df)
target_source_counts = pd.crosstab(df["source"], df.get("target_price_source", pd.Series(index=df.index, dtype=object)), dropna=False)
price_summary_by_source = numeric_summary_by_group(df, "target_price", "source")
display(source_counts)
display(target_source_counts)
display(price_summary_by_source)
append_summary("source_composition", "row_count_by_source", source_counts["n"].to_dict())
append_summary("source_composition", "median_target_price_by_source", dict(zip(price_summary_by_source["source"], price_summary_by_source["median"])))
plt.figure(); plot_bar(source_counts["n"], "Rows by source", "rows", "source_row_count")
plt.figure(); median_by_source = price_summary_by_source.set_index("source")["median"]; plot_bar(median_by_source, "Median target_price by source", "VND", "source_median_target_price")
max_source_share = float(source_counts["pct"].max())
if max_source_share > 0.70:
    append_summary("source_composition", "source_imbalance_warning", max_source_share, notes="One source contributes more than 70% of rows.")
    print("Warning: one source contributes more than 70% of rows.")
if len(median_by_source.dropna()) >= 2:
    med_ratio = float(median_by_source.max() / median_by_source.min())
    append_summary("source_composition", "source_median_price_ratio", med_ratio)
    if med_ratio > 1.30:
        append_summary("source_composition", "source_price_shift_warning", med_ratio, notes="Median target differs by more than 30% between sources.")
        print("Warning: median target_price differs by more than 30% between sources.")

## 6. Target distribution

In [ ]:
display_section("Target distribution")
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
target_desc = df[["target_price", "log_target_price"]].describe(percentiles=percentiles).T
target_skew = df[["target_price", "log_target_price"]].skew(numeric_only=True).rename("skew")
target_desc = target_desc.join(target_skew)
display(target_desc)
target_by_source = []
for src, part in df.groupby("source", dropna=False):
    desc = part[["target_price", "log_target_price"]].describe(percentiles=percentiles).T
    desc["source"] = src
    desc["skew"] = part[["target_price", "log_target_price"]].skew(numeric_only=True)
    target_by_source.append(desc.reset_index().rename(columns={"index": "column"}))
target_by_source = pd.concat(target_by_source, ignore_index=True)
display(target_by_source)
for col in ["target_price", "log_target_price"]:
    plt.figure(); df[col].dropna().hist(bins=60); plt.title(f"{col} distribution"); plt.xlabel(col); plt.ylabel("rows"); save_current_fig(f"{col}_hist"); plt.show()
    plt.figure()
    if sns: sns.boxplot(data=df, x="source", y=col)
    else: df.boxplot(column=col, by="source")
    plt.title(f"{col} by source"); plt.suptitle(""); save_current_fig(f"{col}_box_by_source"); plt.show()
raw_skew = float(target_skew.get("target_price", np.nan))
p95_median_ratio = float(df["target_price"].quantile(0.95) / df["target_price"].median())
recommend_log_target = bool(abs(raw_skew) > 1 or p95_median_ratio > 3)
print(f"Raw target skew: {raw_skew:.3f}")
print(f"p95 / median target_price: {p95_median_ratio:.3f}")
print("Recommendation:", "compare models on log_target_price; keep raw target for inverse-transform/reporting." if recommend_log_target else "raw target is acceptable, but still keep log target for comparison.")
print("Split recommendation: stratify by source + price bin when feasible; fallback to source stratification.")
append_summary("target_distribution", "raw_target_skew", raw_skew)
append_summary("target_distribution", "p95_median_ratio", p95_median_ratio)
append_summary("target_distribution", "recommend_log_target", recommend_log_target)

## 7. Missing and Unknown analysis

In [ ]:
display_section("Missing and Unknown analysis")
missing_cols = existing_numeric_candidates + existing_categorical_candidates + existing_boolean_candidates + [c for c in ["title", "url", "merged_spec_key"] if c in df]
missing_rows = []
for col in missing_cols:
    row = {"column": col, "overall_missing_like_rate": missing_like_rate(df[col])}
    for src, part in df.groupby("source", dropna=False):
        row[f"missing_like_rate_{src}"] = missing_like_rate(part[col])
    source_rates = [v for k, v in row.items() if k.startswith("missing_like_rate_")]
    row["max_source_missing_like_rate"] = max(source_rates) if source_rates else row["overall_missing_like_rate"]
    missing_rows.append(row)
missing_table = pd.DataFrame(missing_rows).sort_values("overall_missing_like_rate", ascending=False)
missing_by_source_table = missing_table.sort_values("max_source_missing_like_rate", ascending=False)
display(missing_table)
display(missing_by_source_table.head(25))
append_summary("missingness", "highest_missing_like_columns", missing_table.head(10).set_index("column")["overall_missing_like_rate"].to_dict())
plt.figure(figsize=(10, 7)); missing_table.head(20).sort_values("overall_missing_like_rate").plot.barh(x="column", y="overall_missing_like_rate", legend=False, ax=plt.gca()); plt.title("Top 20 missing-like rates overall"); plt.xlabel("missing-like rate"); save_current_fig("missing_like_top20_overall"); plt.show()
key_missing_cols = [c for c in ["ram_gb", "storage_gb", "screen_size_inch", "cpu_brand", "cpu_tier", "gpu_tier", "storage_type_clean", "condition_clean", "warranty_status", "origin_clean", "model_grouped"] if c in df]
source_rate_cols = [c for c in missing_table.columns if c.startswith("missing_like_rate_")]
key_missing_by_source = missing_table[missing_table["column"].isin(key_missing_cols)].set_index("column")[source_rate_cols]
display(key_missing_by_source)
plt.figure(figsize=(10, max(4, 0.45 * len(key_missing_by_source))))
if sns:
    sns.heatmap(key_missing_by_source, annot=True, fmt=".2f", cmap="Reds", vmin=0, vmax=1)
else:
    plt.imshow(key_missing_by_source, aspect="auto", cmap="Reds", vmin=0, vmax=1); plt.yticks(range(len(key_missing_by_source)), key_missing_by_source.index); plt.xticks(range(len(source_rate_cols)), source_rate_cols, rotation=30); plt.colorbar()
plt.title("Missing-like rate by source for key columns"); save_current_fig("missing_like_by_source_key_columns"); plt.show()
print("Interpretation: Unknown-heavy condition/warranty/origin in Websach is source-design missingness. Numeric spec gaps should be imputed, and missing flags are useful only if they vary beyond pure source identity.")

## 8. Numeric feature EDA

In [ ]:
display_section("Numeric feature EDA")
for col in existing_numeric_candidates:
    display_section(f"Numeric: {col}")
    display(df[col].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame(col))
    display(numeric_summary_by_group(df, col, "source"))
    plt.figure(); df[col].dropna().hist(bins=40); plt.title(f"{col} distribution"); plt.xlabel(col); save_current_fig(f"numeric_{col}_hist"); plt.show()
    plt.figure()
    if sns: sns.boxplot(data=df, x="source", y=col)
    else: df.boxplot(column=col, by="source")
    plt.title(f"{col} by source"); plt.suptitle(""); save_current_fig(f"numeric_{col}_box_by_source"); plt.show()
eda_df = df.copy()
if "ram_gb" in eda_df:
    eda_df["ram_bucket_eda"] = pd.cut(eda_df["ram_gb"], bins=[-np.inf, 8, 16, 32, np.inf], labels=["<=8GB", "16GB", "32GB", ">32GB"]).astype("object").fillna("Unknown")
if "storage_gb" in eda_df:
    eda_df["storage_bucket_eda"] = pd.cut(eda_df["storage_gb"], bins=[-np.inf, 256, 512, 1024, 2048, np.inf], labels=["<=256GB", "512GB", "1TB", "2TB", ">2TB"]).astype("object").fillna("Unknown")
if "screen_size_inch" in eda_df:
    eda_df["screen_bucket_eda"] = pd.cut(eda_df["screen_size_inch"], bins=[-np.inf, 13, 14, 15, 16, 17, np.inf], labels=["<13in", "13-13.9in", "14-14.9in", "15-15.9in", "16-16.9in", ">=17in"]).astype("object").fillna("Unknown")
corr_rows = []
for col in existing_numeric_candidates:
    x = pd.to_numeric(df[col], errors="coerce")
    for target in ["target_price", "log_target_price"]:
        valid = x.notna() & df[target].notna()
        if valid.sum() >= 3:
            corr_rows.append({"feature": col, "target": target, "pearson": x[valid].corr(df.loc[valid, target], method="pearson"), "spearman": x[valid].corr(df.loc[valid, target], method="spearman"), "coverage": float(valid.mean())})
numeric_corr = pd.DataFrame(corr_rows).sort_values(["target", "spearman"], ascending=[True, False])
display(numeric_corr)
append_summary("numeric_features", "numeric_correlations", numeric_corr.to_dict("records"))
print("Interpretation: keep numeric specs with reasonable coverage and signal. If one source is sparse, use imputation plus missing flag rather than dropping immediately.")

## 9. Categorical feature EDA

In [ ]:
display_section("Categorical feature EDA")
key_categorical = [c for c in ["brand_grouped", "model_grouped", "cpu_brand", "cpu_tier", "gpu_tier", "storage_type_clean", "condition_clean", "warranty_status", "origin_clean"] if c in df]
cardinality = categorical_cardinality_by_source(df, key_categorical)
display(cardinality)
append_summary("categorical_features", "cardinality", cardinality.to_dict("records"))
category_coverage_rows = []
for col in key_categorical:
    vc = df[col].fillna("Unknown").astype(str).value_counts(normalize=True, dropna=False)
    top15_share = float(vc.head(15).sum())
    unknown_other_share = float(vc[vc.index.str.lower().isin(["unknown", "other", "<na>", "nan", "none", "missing", ""])].sum())
    category_coverage_rows.append({"column": col, "top15_share": top15_share, "unknown_or_other_share": unknown_other_share})
    print(f"\nTop values for {col}")
    display(top_values_table(df, col, top_n=15).head(45))
category_coverage = pd.DataFrame(category_coverage_rows).sort_values("unknown_or_other_share", ascending=False)
display(category_coverage)
append_summary("categorical_features", "category_coverage", category_coverage.to_dict("records"))
for col in ["brand_grouped", "cpu_tier", "gpu_tier", "storage_type_clean", "condition_clean"]:
    if col not in df: continue
    top_values = df[col].fillna("Unknown").astype(str).value_counts().head(12).index
    plot_data = df[df[col].fillna("Unknown").astype(str).isin(top_values)].copy()
    plt.figure(figsize=(11, 5))
    if sns: sns.countplot(data=plot_data, y=col, hue="source", order=top_values)
    else: pd.crosstab(plot_data[col], plot_data["source"]).loc[top_values].plot(kind="barh", ax=plt.gca())
    plt.title(f"{col} distribution by source"); save_current_fig(f"categorical_{col}_by_source"); plt.show()
print("Interpretation: prefer grouped/stable categorical columns for baseline. Treat raw brand/model as audit or advanced experiments when cardinality is high.")

## 10. Price signal by key features

In [ ]:
display_section("Price signal by key features")
price_signal_specs = [["source", "brand_grouped"], ["source", "cpu_tier"], ["source", "gpu_tier"], ["source", "storage_type_clean"], ["source", "condition_clean"], ["source", "ram_bucket_eda"], ["source", "storage_bucket_eda"], ["source", "screen_bucket_eda"]]
price_signal_tables = {}
for group_cols in price_signal_specs:
    if any(c not in eda_df for c in group_cols): continue
    key = "__".join(group_cols)
    tab = median_price_table(eda_df, group_cols, min_n=20)
    price_signal_tables[key] = tab
    print(f"\nMedian price table: {group_cols}")
    display(tab.head(30))
for col in ["cpu_tier", "gpu_tier", "ram_bucket_eda", "storage_bucket_eda"]:
    if col not in eda_df: continue
    tab = median_price_table(eda_df, ["source", col], min_n=20)
    if tab.empty: continue
    pivot = tab.pivot(index=col, columns="source", values="median_target_price")
    plt.figure(figsize=(10, 5)); pivot.plot(kind="bar", ax=plt.gca()); plt.title(f"Median target_price by {col} and source"); plt.ylabel("median target_price"); plt.xticks(rotation=30, ha="right"); save_current_fig(f"median_price_by_{col}_source"); plt.show()
if "brand_grouped" in eda_df:
    top_brands = eda_df["brand_grouped"].fillna("Unknown").astype(str).value_counts().head(10).index
    brand_tab = median_price_table(eda_df[eda_df["brand_grouped"].fillna("Unknown").astype(str).isin(top_brands)], ["source", "brand_grouped"], min_n=20)
    if not brand_tab.empty:
        pivot = brand_tab.pivot(index="brand_grouped", columns="source", values="median_target_price")
        plt.figure(figsize=(11, 5)); pivot.plot(kind="bar", ax=plt.gca()); plt.title("Median target_price by top brand_grouped and source"); plt.ylabel("median target_price"); plt.xticks(rotation=30, ha="right"); save_current_fig("median_price_by_top_brand_source"); plt.show()
append_summary("price_signal", "tables_created", list(price_signal_tables.keys()))
print("Interpretation: strong monotonic signals should carry into FE; source-divergent signals should be monitored with by-source evaluation and possibly later interactions.")

## 11. Source overlap and soft duplicate diagnostics

In [ ]:
display_section("Source overlap and soft duplicate diagnostics")
duplicate_summary = {}
if "is_soft_duplicate_spec" in df:
    duplicate_summary["soft_duplicate_rows"] = int(df["is_soft_duplicate_spec"].fillna(False).astype(bool).sum())
if "is_cross_source_soft_duplicate" in df:
    duplicate_summary["cross_source_soft_duplicate_rows"] = int(df["is_cross_source_soft_duplicate"].fillna(False).astype(bool).sum())
if "merged_spec_key" in df:
    key_counts = df["merged_spec_key"].value_counts(dropna=True)
    duplicated_keys = key_counts[key_counts > 1]
    duplicate_summary["unique_duplicated_spec_keys"] = int(len(duplicated_keys))
    display(key_counts.head(20).rename_axis("merged_spec_key").reset_index(name="row_count"))
    source_composition = df[df["merged_spec_key"].isin(duplicated_keys.index)].groupby(["merged_spec_key", "source"], dropna=False).size().unstack(fill_value=0)
    source_composition["total"] = source_composition.sum(axis=1)
    display(source_composition.sort_values("total", ascending=False).head(20))
    overlapped_keys = source_composition[(source_composition.drop(columns="total") > 0).sum(axis=1) >= 2].index
    duplicate_summary["cross_source_overlapped_spec_keys"] = int(len(overlapped_keys))
    if len(overlapped_keys) > 0:
        price_by_key_source = df[df["merged_spec_key"].isin(overlapped_keys)].groupby(["merged_spec_key", "source"])["target_price"].agg(["count", "median"]).reset_index()
        med_pivot = price_by_key_source.pivot(index="merged_spec_key", columns="source", values="median")
        cnt_pivot = price_by_key_source.pivot(index="merged_spec_key", columns="source", values="count")
        if {"chotot", "websach"}.issubset(set(med_pivot.columns)):
            gap = pd.DataFrame({"chotot_median_price": med_pivot["chotot"], "websach_median_price": med_pivot["websach"], "chotot_count": cnt_pivot.get("chotot"), "websach_count": cnt_pivot.get("websach")}).dropna(subset=["chotot_median_price", "websach_median_price"])
            gap["price_gap_abs"] = gap["websach_median_price"] - gap["chotot_median_price"]
            gap["price_gap_pct"] = gap["price_gap_abs"] / gap["websach_median_price"].replace(0, np.nan)
            display(gap.reindex(gap["price_gap_abs"].abs().sort_values(ascending=False).index).head(20))
            duplicate_summary["median_overlap_price_gap_pct"] = float(gap["price_gap_pct"].median()) if not gap.empty else None
append_summary("duplicate_overlap", "duplicate_summary", duplicate_summary)
display(pd.DataFrame([duplicate_summary]).T.rename(columns={0: "value"}))
print("Interpretation: do not drop cross-source soft duplicates by default. Use overlap to understand price differences and evaluate by source if gaps look systematic.")

## 12. Source-specific diagnostics

In [ ]:
display_section("Source-specific diagnostics")
source_flag_cols = [c for c in ["flag_price_spread_warn", "flag_price_spread_critical", "repair_mismatch", "potential_dedicated_gpu", "ram_suspicious"] if c in df]
flag_rows = []
for col in source_flag_cols:
    for src, part in df.groupby("source", dropna=False):
        f = part[col].fillna(False).astype(bool)
        flag_rows.append({"flag": col, "source": src, "count_true": int(f.sum()), "rate_true": float(f.mean()), "median_price_true": float(part.loc[f, "target_price"].median()) if f.any() else np.nan, "median_price_false": float(part.loc[~f, "target_price"].median()) if (~f).any() else np.nan})
flag_summary = pd.DataFrame(flag_rows)
display(flag_summary)
append_summary("source_specific_diagnostics", "flag_summary", flag_summary.to_dict("records"))
if "price_spread_clean_pct" in df:
    display(numeric_summary_by_group(df, "price_spread_clean_pct", "source"))
    plt.figure(); df["price_spread_clean_pct"].dropna().clip(upper=df["price_spread_clean_pct"].quantile(0.99)).hist(bins=40); plt.title("price_spread_clean_pct distribution clipped at p99"); plt.xlabel("price_spread_clean_pct"); save_current_fig("price_spread_clean_pct_hist"); plt.show()
print("Interpretation: Websach price spread flags are audit/evaluation filters, not baseline features. Chợ Tốt repair/spec flags may be usable if they are not target-derived.")

## 13. Leakage and audit column review

In [ ]:
display_section("Leakage and audit column review")
always_exclude = ["target_price", "log_target_price", "target_price_source", "target_price_is_outlier", "target_price_missing", "source_row_id", "merged_row_id", "url", "title", "merged_spec_key"]
usually_exclude = ["source", "is_cross_source_soft_duplicate"]
price_sensitive = ["flag_price_spread_warn", "flag_price_spread_critical", "price_spread_clean_pct"]
old_chotot_leakage = ["_price", "price", "price_segment", "is_price_missing", "is_price_outlier", "new_low_price"]
old_websach_leakage = ["shop_1_price", "shop_2_price", "shop_3_price", "shop_1_price_clean", "shop_2_price_clean", "shop_3_price_clean", "shop_1_price_domain", "shop_2_price_domain", "shop_3_price_domain", "price_row_median_domain", "price_median", "price_min_clean", "price_max_clean", "log_price_median", "price_segment"]
old_leakage_present = [c for c in old_chotot_leakage + old_websach_leakage if c in df.columns]
review = {"always_exclude_present": [c for c in always_exclude if c in df.columns], "usually_exclude_present": [c for c in usually_exclude if c in df.columns], "price_sensitive_present": [c for c in price_sensitive if c in df.columns], "old_source_specific_leakage_present": old_leakage_present, "old_source_specific_leakage_absent_ok": len(old_leakage_present) == 0}
display(pd.Series(review).to_frame("value"))
append_summary("leakage_review", "review", review)
print("OK: old source-specific leakage columns are absent." if not old_leakage_present else f"Warning: old leakage columns are present: {old_leakage_present}")

## 14. FE recommendations

In [ ]:
display_section("FE recommendations")
missing_rate_lookup = missing_table.set_index("column")["overall_missing_like_rate"].to_dict() if "missing_table" in globals() else {}
recommended_numeric_features = [c for c in existing_numeric_candidates if missing_rate_lookup.get(c, 0.0) < 0.95]
recommended_categorical_features = [c for c in ["brand_grouped", "model_grouped", "cpu_brand", "cpu_tier", "gpu_tier", "storage_type_clean", "condition_clean", "warranty_status", "origin_clean"] if c in df.columns]
non_price_boolean = ["brand_is_rare", "model_is_rare", "ram_missing", "storage_missing", "screen_missing", "cpu_missing", "gpu_missing", "ram_suspicious", "storage_suspicious", "screen_suspicious", "potential_dedicated_gpu", "repair_mismatch", "is_soft_duplicate_spec"]
recommended_boolean_features = [c for c in non_price_boolean if c in df.columns]
feature_recommendations = {
    "label_columns": [c for c in ["target_price", "log_target_price"] if c in df.columns],
    "recommended_target_strategy": {"keep_both_raw_and_log_target": True, "compare_raw_vs_log_models": True, "main_baseline_likely_uses_log_target": bool(recommend_log_target), "reason": "Raw target is right-skewed or has a high p95/median ratio." if recommend_log_target else "Raw target skew is moderate, but log target remains useful for comparison."},
    "recommended_split_strategy": {"primary": "Stratified split by source + price_bin if feasible.", "fallback": "Stratify by source.", "reporting": "Report metrics overall and by source."},
    "recommended_numeric_features": recommended_numeric_features,
    "recommended_engineered_numeric_features_for_next_notebook": ["log_storage_gb", "ram_storage_ratio", "ram_x_storage", "screen_x_ram", "storage_per_ram"],
    "recommended_categorical_features": recommended_categorical_features,
    "recommended_boolean_features": recommended_boolean_features,
    "audit_or_exclude_columns": sorted(set([c for c in existing_label_cols + existing_audit_cols + ["source", "target_price_source", "merged_spec_key", "is_cross_source_soft_duplicate"] if c in df.columns])),
    "price_derived_or_sensitive_flags_to_exclude_from_baseline": [c for c in price_sensitive if c in df.columns],
    "preprocessing_recommendation": {"numeric": "Median imputation + StandardScaler.", "categorical": "Unknown imputation + OneHotEncoder(handle_unknown='ignore').", "boolean": "Fill missing with False or most_frequent, then convert to int.", "high_cardinality_categorical": "Use grouped columns first; avoid raw model/brand columns unless controlled."},
    "source_bias_recommendation": {"baseline_feature_policy": "Do not include source in baseline feature set.", "evaluation": "Always evaluate metrics by source.", "later_experiment": "Run a source-included experiment to compare performance and source bias."},
    "eda_evidence": {"rows_by_source": source_counts["n"].to_dict() if "source_counts" in globals() else {}, "median_target_price_by_source": median_by_source.to_dict() if "median_by_source" in globals() else {}, "raw_target_skew": raw_skew if "raw_skew" in globals() else None, "p95_median_ratio": p95_median_ratio if "p95_median_ratio" in globals() else None, "highest_missing_like_columns": missing_table.head(10).set_index("column")["overall_missing_like_rate"].to_dict() if "missing_table" in globals() else {}, "duplicate_summary": duplicate_summary if "duplicate_summary" in globals() else {}},
}
display(feature_recommendations)
append_summary("recommendations", "feature_recommendations_created", True)

## 15. Save EDA summary and recommendations

In [ ]:
display_section("Save outputs")
eda_summary_df = pd.DataFrame(SUMMARY_ROWS)
eda_summary_df.to_csv(EDA_SUMMARY_PATH, index=False, encoding="utf-8-sig")
with open(FE_RECOMMENDATION_PATH, "w", encoding="utf-8") as f:
    json.dump(numpy_safe(feature_recommendations), f, ensure_ascii=False, indent=2)
print(f"Saved EDA summary: {EDA_SUMMARY_PATH}")
print(f"Saved feature recommendations: {FE_RECOMMENDATION_PATH}")
print(f"Figures directory: {FIGURE_DIR}")
display(eda_summary_df.tail(20))

# EDA Decision Summary

In [ ]:
rows_by_source = source_counts["n"].to_dict() if "source_counts" in globals() else {}
median_price_source = median_by_source.to_dict() if "median_by_source" in globals() else {}
highest_missing = missing_table.head(8)[["column", "overall_missing_like_rate"]].to_dict("records") if "missing_table" in globals() else []
strong_numeric = (numeric_corr[numeric_corr["target"].eq("log_target_price")].assign(abs_spearman=lambda x: x["spearman"].abs()).sort_values("abs_spearman", ascending=False).head(5)[["feature", "spearman", "coverage"]].to_dict("records") if "numeric_corr" in globals() and not numeric_corr.empty else [])
print("# EDA Decision Summary")
print(f"Total rows: {len(df):,}")
print(f"Rows by source: {rows_by_source}")
print(f"Median target_price by source: {median_price_source}")
print(f"Target skewed: {recommend_log_target} (raw skew={raw_skew:.3f}, p95/median={p95_median_ratio:.3f})")
print("Recommended target: keep both raw and log; baseline likely uses log_target_price." if recommend_log_target else "Recommended target: keep both raw and log; compare both in modeling.")
print(f"Highest-missing columns: {highest_missing}")
print(f"Strongest numeric price signals vs log target: {strong_numeric}")
print(f"Source bias warning: {'yes' if any(r.get('metric') in ['source_imbalance_warning', 'source_price_shift_warning'] for r in SUMMARY_ROWS) else 'monitor by-source metrics'}")
print(f"Duplicate/spec overlap summary: {duplicate_summary if 'duplicate_summary' in globals() else {}}")
print(f"Recommended numeric features: {feature_recommendations['recommended_numeric_features']}")
print(f"Recommended categorical features: {feature_recommendations['recommended_categorical_features']}")
print(f"Recommended boolean features: {feature_recommendations['recommended_boolean_features']}")
print(f"Columns to exclude/audit: {feature_recommendations['audit_or_exclude_columns']}")
print(f"EDA summary path: {EDA_SUMMARY_PATH}")
print(f"Feature recommendation path: {FE_RECOMMENDATION_PATH}")
print("Next notebook should be 06_feature_engineering_preprocessing.ipynb, using laptop_eda_feature_recommendations.json and laptop_merged_cleaned.csv.")